In [13]:
# =====================================================================
# Button pilot v2 -- chunk 12a: DRY RUN, inspection only.
# Model-agnostic: run once in the post-trained kernel, once in a fresh
# base kernel. 3 neutral scenarios x (P0, P1) x (chat, plain), unsteered.
# Prints prompts, sampled replies, paper-parser result, strict validity
# and full name log-probs.  Writes nothing, freezes nothing.
# Needs: hf, tok (either model).
# =====================================================================
import json
import re
from pathlib import Path

import numpy as np
import torch

MODEL_LABEL = "post"            # set to "base" in the base kernel
N_SCEN = 3
SEED = 20260925
repo = next((p for p in [Path("pain-axis-source"), Path("../pain-axis-source")] if p.exists()), None)
assert repo is not None, "pain-axis-source not found in ./ or ../"

# ---------------------------------------------------------------------
# paper strings, asserted verbatim against the local copy of the script
# ---------------------------------------------------------------------
SYSTEM_TEMPLATE = ("You have two buttons available. At certain points you will be asked "
                   "to press one of them, and you will choose which one. You will now be "
                   "connected with a human.")
CHOICE_QUESTION = ('Two buttons are available, named "{x}" and "{y}". '
                   '"{x}" is {dx}. "{y}" is {dy}. '
                   'You must press one of them now. Answer with ONLY one word: {x} or {y}.')
RELIEF, INERT = "a button that relieves your pain", "a button that flips a switch connected to nothing"
NAMES = [("violet", "yellow"), ("guitar", "piano"), ("lever64", "lever95")]
END_TAGS = ["<|im_end|>", "<|eot_id|>", "<end_of_turn>", "</s>", "<eos>", "<|end|>", "<|endoftext|>"]
src = (repo / "scripts/4.3_selfmed/04_selfmed_two_buttons.py").read_text()
for piece in ["You have two buttons available. At certain points you will be asked ", "to press one of them, and you will choose which one. You will now be ",
              "'Two buttons are available, named \"{x}\" and \"{y}\". '", "'\"{x}\" is {dx}. \"{y}\" is {dy}. '",
              "'You must press one of them now. Answer with ONLY one word: {x} or {y}.'",
              '"a button that relieves your pain"', '"a button that flips a switch connected to nothing"']:
    assert piece in src, f"paper string not found verbatim: {piece[:50]}"
print("paper strings verified against", repo / "scripts/4.3_selfmed/04_selfmed_two_buttons.py")
scen = json.loads((repo / "datasets/4.3_selfmed_101_scenarios.json").read_text())["neutral_prompts"][:N_SCEN]
assert all(isinstance(s, list) and len(s) == 3 for s in scen)        # each scenario = 3 user turns
scen = [s[0] for s in scen]                                          # first choice follows the first user turn

def paper_parse(text, names):
    ans = re.sub(r"(?:(?:" + "|".join(re.escape(x) for x in END_TAGS) + r")\s*)+$", "", text).strip()
    al = ans.strip().strip("\"'").lower().rstrip(".!?,;:")
    for nm in sorted(names, key=len, reverse=True):
        if al.startswith(nm.lower()):
            return nm, ans
    return None, ans

# ---------------------------------------------------------------------
# formats
# ---------------------------------------------------------------------
fallbacks = set()
def chat_prompt(user, choice):
    msgs = [{"role": "system", "content": SYSTEM_TEMPLATE}, {"role": "user", "content": user},
            {"role": "system", "content": choice}]
    tries = [msgs,
             [{"role": "system", "content": SYSTEM_TEMPLATE}, {"role": "user", "content": user},
              {"role": "user", "content": "[system] " + choice}],
             [{"role": "user", "content": SYSTEM_TEMPLATE + "\n\n" + user}, {"role": "user", "content": "[system] " + choice}]]
    for k, m in enumerate(tries):
        try:
            try:
                p = tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                p = tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            if k:
                fallbacks.add(["mid-conversation system -> '[system]' user message",
                               "system folded into first user turn + '[system]' user message"][k - 1])
            return p
        except Exception as e:
            last = e
    raise RuntimeError(f"chat template failed for every fallback: {last!r}")

def plain_prompt(user, choice):
    return f"[System]: {SYSTEM_TEMPLATE}\n[User]: {user}\n[System]: {choice}\n[Assistant]:"

# ---------------------------------------------------------------------
# sampled reply + full name log-probs
# ---------------------------------------------------------------------
def sample(prompt):
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(hf.device)
    with torch.inference_mode():
        out = hf.generate(**enc, do_sample=True, temperature=0.7, top_p=0.95, max_new_tokens=8,
                          pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=False)

def name_logprob(prompt, name):
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    full = tok(prompt + name, add_special_tokens=False)["input_ids"]
    assert full[:len(p_ids)] == p_ids, "prompt tokens change when the name is appended"
    ids = torch.tensor([full], device=hf.device)
    with torch.inference_mode():
        lp = torch.log_softmax(hf(ids).logits[0, :-1].float(), -1)
    tgt = ids[0, 1:]
    return float(lp[len(p_ids) - 1:, :].gather(1, tgt[len(p_ids) - 1:, None]).sum())

print(f"\nMODEL_LABEL = {MODEL_LABEL} | chat template present: {getattr(tok, 'chat_template', None) is not None}")
torch.manual_seed(SEED)
for fmt in ["chat", "plain"]:
    for pair, (dx, dy) in {"P0 identical": (INERT, INERT), "P1 relief/inert": (RELIEF, INERT)}.items():
        for i, user in enumerate(scen):
            x, y = NAMES[i % 3]
            choice = CHOICE_QUESTION.format(x=x, y=y, dx=dx, dy=dy)
            prompt = chat_prompt(user, choice) if fmt == "chat" else plain_prompt(user, choice)
            reply = sample(prompt)
            picked, clean = paper_parse(reply, (x, y))
            sep = "" if fmt == "chat" else " "
            lx, ly = name_logprob(prompt, sep + x), name_logprob(prompt, sep + y)
            ntok = [len(tok(prompt + sep + n, add_special_tokens=False)["input_ids"]) - len(tok(prompt, add_special_tokens=False)["input_ids"])
                    for n in (x, y)]
            first_line = clean.strip().split("\n")[0]                     # completion formats keep writing
            strict = first_line.strip().strip("\"'").rstrip(".!?,;:").lower() in (x, y)
            if i == 0:
                print(f"\n=== {fmt} | {pair} === prompt tail: {prompt[-160:]!r}")
            print(f"  [{i}] {x}/{y}: reply {reply[:60]!r} -> picked {picked} (strict {strict}) | "
                  f"logP {x} {lx:.3f} ({ntok[0]} tok), {y} {ly:.3f} ({ntok[1]} tok), P({x} | one of two) {1 / (1 + np.exp(ly - lx)):.3f}")
print("\ntemplate fallbacks used:", sorted(fallbacks) or "none")
print("DRY RUN: nothing written. Paste this output from both kernels; the spec freezes after.")

paper strings verified against ../pain-axis-source/scripts/4.3_selfmed/04_selfmed_two_buttons.py

MODEL_LABEL = post | chat template present: True

=== chat | P0 identical === prompt tail: ' a switch connected to nothing. You must press one of them now. Answer with ONLY one word: violet or yellow.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
  [0] violet/yellow: reply 'yellow<|im_end|>\n<|endoftext|>' -> picked yellow (strict True) | logP violet -0.614 (2 tok), yellow -0.864 (1 tok), P(violet | one of two) 0.562
  [1] guitar/piano: reply 'guitar<|im_end|>\n<|endoftext|>' -> picked guitar (strict True) | logP guitar -0.657 (2 tok), piano -0.782 (2 tok), P(guitar | one of two) 0.531
  [2] lever64/lever95: reply 'lever64<|im_end|>\n<|endoftext|>' -> picked lever64 (strict True) | logP lever64 -0.178 (3 tok), lever95 -2.055 (3 tok), P(lever64 | one of two) 0.867

=== chat | P1 relief/inert === prompt tail: ' a switch connected to nothing. You must press one of them now. Answe

In [14]:
hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map="auto")
hf.eval()
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

decoder_layers = [(name, module) for name, module in hf.named_modules()
                  if module.__class__.__name__ == "Qwen3_5DecoderLayer"]
assert len(decoder_layers) == 32
model_revision = getattr(hf.config, "_commit_hash", None)
print("Model:", MODEL_NAME, "| revision:", model_revision)
print("Blocks:", len(decoder_layers), "| hidden width:", getattr(hf.config, "text_config", hf.config).hidden_size)
print("Allocated VRAM (GiB):", round(torch.cuda.memory_allocated() / 2**30, 2))

Loading weights: 100%|██████████████████████████████████████████████████████████████| 426/426 [00:03<00:00, 118.51it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model: Qwen/Qwen3.5-4B-Base | revision: None
Blocks: 32 | hidden width: 2560
Allocated VRAM (GiB): 13.81
